# Week 15 — QuakeMigrate Validation
Benchmark comparison against Illsley-Kemp and Mestel (2025), reusing the
exact approach from Weeks 7 (EQTransformer) and 11 (PhaseNet). Also
extends Week 11's two-way comparison into a full three-way comparison
across all three catalogues, now that QuakeMigrate's is complete too.
Given the small sample size (31 events), Mc/b-value results are computed
for comparability but should be read with real caution - noted explicitly
in the interpretation, not just in this comment.

In [ ]:
import os
import numpy as np
import pandas as pd
from obspy.geodetics import gps2dist_azimuth

PROJECT_DIR = os.path.expanduser("~/tvz_project")
CATALOG_DIR = os.path.join(PROJECT_DIR, "catalogs")
BENCH_DIR = os.path.join(CATALOG_DIR, "benchmark")

WINDOW_START = pd.Timestamp("2019-08-01T00:00:00")
WINDOW_END = pd.Timestamp("2019-09-30T23:59:59")
TIME_TOL_S = 2.0
DIST_TOL_KM = 5.0

In [ ]:
# Load the QuakeMigrate catalogue (already has magnitudes - computed
# directly during Locate, not a separate step like EQT/PhaseNet)
ours = pd.read_csv(os.path.join(CATALOG_DIR, "quakemigrate_quality_filtered.csv"))
ours["origin_dt"] = pd.to_datetime(ours["origin_time"])
ours = ours.rename(columns={"ml": "ml_median"})  # match column name used in other catalogues' comparisons

print(f"QuakeMigrate catalogue: {len(ours)} events")
print(ours[["origin_dt", "lat", "lon", "depth_km", "ml_median"]])

In [ ]:
bench = pd.read_csv(os.path.join(BENCH_DIR, "absolute_locations.csv"))
bench["origin_dt"] = pd.to_datetime(bench["origintime"])
bench_window = bench[(bench["origin_dt"] >= WINDOW_START) & (bench["origin_dt"] <= WINDOW_END)].copy()

print(f"Benchmark events in our window: {len(bench_window)}")
print(f"QuakeMigrate catalogue (quality-filtered): {len(ours)} events")
print(f"Event count ratio (ours / benchmark): {len(ours) / len(bench_window):.3f}")

In [ ]:
def find_matches(source_df, target_df, time_tol_s, dist_tol_km):
    matched_flags = np.zeros(len(source_df), dtype=bool)
    match_dist_km = np.full(len(source_df), np.nan)

    target_times = target_df["origin_dt"].values
    for i, row in enumerate(source_df.itertuples()):
        dt_seconds = np.abs((target_times - np.datetime64(row.origin_dt)) / np.timedelta64(1, "s"))
        candidates = np.where(dt_seconds <= time_tol_s)[0]
        if len(candidates) == 0:
            continue
        best_dist = np.inf
        for c in candidates:
            trow = target_df.iloc[c]
            dist_m, _, _ = gps2dist_azimuth(row.lat, row.lon, trow["latitude"], trow["longitude"])
            dist_km = dist_m / 1000.0
            if dist_km < best_dist:
                best_dist = dist_km
        if best_dist <= dist_tol_km:
            matched_flags[i] = True
            match_dist_km[i] = best_dist

    return matched_flags, match_dist_km

ours_matched, ours_dist = find_matches(ours, bench_window, TIME_TOL_S, DIST_TOL_KM)
ours["matched_to_benchmark"] = ours_matched
ours["match_dist_km"] = ours_dist

recall_input = bench_window.rename(columns={"latitude": "lat", "longitude": "lon"})
bench_matched, _ = find_matches(recall_input, ours.rename(columns={"lat": "latitude", "lon": "longitude"}), TIME_TOL_S, DIST_TOL_KM)
bench_window = bench_window.copy()
bench_window["matched_to_ours"] = bench_matched

precision = ours_matched.sum() / len(ours)
recall = bench_matched.sum() / len(bench_window)

print(f"Precision (fraction of QuakeMigrate's {len(ours)} events matching benchmark): {precision:.3f}")
print(f"Recall (fraction of benchmark's {len(bench_window)} events found): {recall:.3f}")
print(f"\nFor comparison: EQTransformer precision 0.612/recall 0.537; PhaseNet precision 0.571/recall 0.516")

In [ ]:
def maxc_mc(magnitudes, bin_width=0.1):
    mags = magnitudes.dropna().values
    bins = np.arange(np.floor(mags.min()), np.ceil(mags.max()) + bin_width, bin_width)
    counts, edges = np.histogram(mags, bins=bins)
    peak_idx = np.argmax(counts)
    return edges[peak_idx] + bin_width / 2

def mle_bvalue(magnitudes, mc, bin_width=0.1):
    mags = magnitudes.dropna().values
    above_mc = mags[mags >= mc - bin_width / 2]
    mean_m = above_mc.mean()
    b = np.log10(np.e) / (mean_m - (mc - bin_width / 2))
    return b, len(above_mc)

mc_ours = maxc_mc(ours["ml_median"])
b_ours, n_ours = mle_bvalue(ours["ml_median"], mc_ours)

mc_bench = maxc_mc(bench_window["magnitude"])
b_bench, n_bench = mle_bvalue(bench_window["magnitude"], mc_bench)

print("=== QuakeMigrate catalogue ===")
print(f"Mc (MAXC): {mc_ours:.2f}, b-value (MLE): {b_ours:.3f} (n={n_ours} of {len(ours)} total - CAUTION: small sample)")
print("\n=== Benchmark catalogue (same window) ===")
print(f"Mc (MAXC): {mc_bench:.2f}, b-value (MLE): {b_bench:.3f} (n={n_bench})")
print("\nFor comparison: EQTransformer Mc=1.55/b=1.474 (n=380); PhaseNet Mc=1.45/b=1.594 (n=290)")

In [ ]:
def spatial_summary(df, lat_col, lon_col, depth_col, label):
    print(f"--- {label} ---")
    print(f"Lat:   mean {df[lat_col].mean():.4f}, std {df[lat_col].std():.4f}")
    print(f"Lon:   mean {df[lon_col].mean():.4f}, std {df[lon_col].std():.4f}")
    print(f"Depth: mean {df[depth_col].mean():.2f} km, std {df[depth_col].std():.2f} km")
    print()

spatial_summary(ours, "lat", "lon", "depth_km", "QuakeMigrate catalogue (all)")
spatial_summary(bench_window, "latitude", "longitude", "depth", "Benchmark catalogue (all, this window)")

ours_unmatched = ours[~ours["matched_to_benchmark"]]
bench_unmatched = bench_window[~bench_window["matched_to_ours"]]
if len(ours_unmatched) > 0:
    spatial_summary(ours_unmatched, "lat", "lon", "depth_km", "QuakeMigrate events NOT found in benchmark")
else:
    print("--- QuakeMigrate events NOT found in benchmark ---\n(none - every QuakeMigrate event matched)\n")
spatial_summary(bench_unmatched, "latitude", "longitude", "depth", "Benchmark events we MISSED")

In [ ]:
# Full three-way comparison, now that all three catalogues are complete
eqt = pd.read_csv(os.path.join(CATALOG_DIR, "eqtransformer_nlloc_quality_filtered.csv"))
eqt["event_idx"] = eqt["file"].str.extract(r"_E(\d+)\.")[0].astype(int)
eqt["origin_dt"] = pd.to_datetime(eqt["origin_time"])
eqt_mags = pd.read_csv(os.path.join(CATALOG_DIR, "eqtransformer_magnitudes.csv"))
eqt = eqt.merge(eqt_mags, on="event_idx", how="left")

pn = pd.read_csv(os.path.join(CATALOG_DIR, "phasenet_nlloc_quality_filtered.csv"))
pn["event_idx"] = pn["file"].str.extract(r"_E(\d+)\.")[0].astype(int)
pn["origin_dt"] = pd.to_datetime(pn["origin_time"])
pn_mags = pd.read_csv(os.path.join(CATALOG_DIR, "phasenet_magnitudes.csv"))
pn = pn.merge(pn_mags, on="event_idx", how="left")

print(f"EQTransformer: {len(eqt)} events, PhaseNet: {len(pn)} events, QuakeMigrate: {len(ours)} events")

qm_vs_eqt, _ = find_matches(ours, eqt.rename(columns={"lat": "latitude", "lon": "longitude"}), TIME_TOL_S, DIST_TOL_KM)
qm_vs_pn, _ = find_matches(ours, pn.rename(columns={"lat": "latitude", "lon": "longitude"}), TIME_TOL_S, DIST_TOL_KM)

print(f"\nQuakeMigrate events also found by EQTransformer: {qm_vs_eqt.sum()} of {len(ours)} ({100*qm_vs_eqt.mean():.1f}%)")
print(f"QuakeMigrate events also found by PhaseNet:      {qm_vs_pn.sum()} of {len(ours)} ({100*qm_vs_pn.mean():.1f}%)")
print(f"\nFor reference (Week 11): {68.7}% of EQTransformer's events were also found by PhaseNet")

In [ ]:
summary_path = os.path.join(BENCH_DIR, "week15_quakemigrate_comparison_summary.csv")
summary = pd.DataFrame([{
    "quakemigrate_event_count": len(ours),
    "benchmark_event_count_window": len(bench_window),
    "precision": precision,
    "recall": recall,
    "mc_quakemigrate": mc_ours,
    "b_value_quakemigrate": b_ours,
    "mc_benchmark": mc_bench,
    "b_value_benchmark": b_bench,
    "qm_events_also_in_eqt_pct": 100 * qm_vs_eqt.mean(),
    "qm_events_also_in_phasenet_pct": 100 * qm_vs_pn.mean(),
}])
summary.to_csv(summary_path, index=False)
print(f"Saved: {summary_path}")

ours.to_csv(os.path.join(BENCH_DIR, "quakemigrate_with_match_flags.csv"), index=False)
bench_window.to_csv(os.path.join(BENCH_DIR, "benchmark_window_quakemigrate_match_flags.csv"), index=False)
print("Saved matched catalogues")